# 01 Data Ingestion Orchestrator (TPC-H to Iceberg via Trino)

Notebook ini berfungsi sebagai orkestrator untuk memindahkan data mentah TPC-H (CSV) ke dalam arsitektur Data Lakehouse (Apache Iceberg) menggunakan **Trino** sebagai *query engine*.

**Langkah-langkah yang akan dijalankan:**
1. **Prerequisite & Sanity Check:** Memastikan Docker (MinIO, Hive, Trino) berjalan.
2. **Upload CSV:** Memuat data CSV lokal ke bucket MinIO.
3. **Cleanup:** Membersihkan sisa data Iceberg sebelumnya agar tidak terjadi duplikasi.
4. **Execute Trino SQL:** Mengeksekusi file `tpch_iceberg_schema.sql` untuk membuat skema, *external table*, dan tabel Iceberg.
5. **Validation:** Menghitung jumlah baris data yang berhasil masuk ke Iceberg.

In [ ]:
import sys
import subprocess
import time
from pathlib import Path

try:
    from minio import Minio
    from trino.dbapi import connect
    print("Library 'minio' dan 'trino' siap digunakan")
except ImportError:
    print("Menginstal library yang dibutuhkan...")
    !pip install minio trino
    from minio import Minio
    from trino.dbapi import connect

# Konfigurasi Koneksi
MINIO_ENDPOINT = "localhost:9000"
MINIO_ACCESS_KEY = "admin"
MINIO_SECRET_KEY = "admin123"

TRINO_HOST = "localhost"
TRINO_PORT = 8080
TRINO_USER = "trino"

Library 'minio' dan 'trino' siap digunakan!


## 1. Sanity Check (Prerequisites Validation)
Memeriksa apakah *container* Docker untuk `minio`, `hive-metastore`, dan `trino` sudah berjalan dengan baik.

In [2]:
def check_docker_service(service_name: str) -> bool:
    # cek apakah container docker sedang berjalan
    try:
        result = subprocess.run(
            ["docker", "ps", "--filter", f"name={service_name}", "--format", "{{.Names}}"],
            capture_output=True, text=True, check=True
        )
        return service_name in result.stdout
    except Exception as e:
        print(f"Error checking docker: {e}")
        return False

services = ["minio", "hive-metastore", "trino"]
all_running = True

for service in services:
    if check_docker_service(service):
        print(f"[OK] {service:20} sedang berjalan")
    else:
        print(f"[X]  {service:20} TIDAK berjalan!")
        all_running = False

if not all_running:
    print("\nPERINGATAN: Ada service yang belum jalan. Jalankan 'docker-compose up -d' dulu")
else:
    print("\nSemua service prerequisites sudah siap.")

[OK] minio                sedang berjalan
[OK] hive-metastore       sedang berjalan
[OK] trino                sedang berjalan

Semua service prerequisites sudah siap.


## 2. Upload CSV & Cleanup Previous Data
Menjalankan script upload CSV ke MinIO dan membersihkan *bucket* `iceberg` dari data eksperimen sebelumnya agar lokasi tabel benar-benar bersih.

In [4]:
# Upload CSV ke MinIO (Memanggil script eksternal)
print("Uploading CSV to MinIO...")
upload_result = subprocess.run(["python", "../code/upload_csv_to_lakehouse.py"], capture_output=True, text=True)
print(upload_result.stdout)
if upload_result.stderr:
    print(f"Errors (if any): {upload_result.stderr}")

# Cleanup Iceberg Bucket di MinIO
print("\nCleaning up previous Iceberg objects...")
client = Minio(
    endpoint=MINIO_ENDPOINT,
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=False
)

bucket_name = "iceberg"
table_prefixes = [f"tpch/{t}/" for t in ["customer", "lineitem", "nation", "orders", "part", "partsupp", "region", "supplier"]]

if client.bucket_exists(bucket_name):
    removed_objects = 0
    for prefix in table_prefixes:
        # cari file dengan prefix tersebut
        objects_to_delete = [obj.object_name for obj in client.list_objects(bucket_name, prefix=prefix, recursive=True)]
        if objects_to_delete:
            for obj_name in objects_to_delete:
                client.remove_object(bucket_name, obj_name)
            removed_objects += len(objects_to_delete)
            print(f"  [-] Menghapus {len(objects_to_delete)} file dari prefix {prefix}")
    
    print(f"Total file Iceberg lama yang dihapus: {removed_objects} files.")
else:
    print(f"Bucket '{bucket_name}' belum ada, melewati proses cleanup.")

Uploading CSV to MinIO...
Project directory: /mnt/c/Users/Muzaraar/Documents/Kuliah/Kelas_Kuliah/Semester_6/BigData/BigData-Kelompok5
CSV directory:     /mnt/c/Users/Muzaraar/Documents/Kuliah/Kelas_Kuliah/Semester_6/BigData/BigData-Kelompok5/data/csv

Connecting to MinIO localhost:9000...
Connected
  Bucket exists: lakehouse

Found 8 CSV file(s)
Uploading to s3://lakehouse/csv/

Cleaning up old files in s3://lakehouse/csv/...
No old objects found.

  Uploaded customer.csv         (  23.38 MB) → csv/customer/customer.csv
  Uploaded lineitem.csv         ( 725.75 MB) → csv/lineitem/lineitem.csv
  Uploaded nation.csv           (   0.00 MB) → csv/nation/nation.csv
  Uploaded orders.csv           ( 164.47 MB) → csv/orders/orders.csv
  Uploaded part.csv             (  23.04 MB) → csv/part/part.csv
  Uploaded partsupp.csv         ( 114.04 MB) → csv/partsupp/partsupp.csv
  Uploaded region.csv           (   0.00 MB) → csv/region/region.csv
  Uploaded supplier.csv         (   1.35 MB) → csv/suppl